# 🎬 Lecture Agent Studio v4

v2에서 발견된 문제점(스크립트 반복 표현, 중복 코드, FFmpeg 경로 불일치 등)을 전면 개선한 버전입니다.

## 📦 Cell 1. 통합 Import & 환경 설정

v2에서 파일 곳곳에 흩어져 있던 import문과 환경변수를 **한 곳에 통합**했습니다.

In [1]:
# ============================================================
# 📦 통합 Import & 환경 설정 (v3 개선: 중복 제거, 한 곳에 집중)
# ============================================================
import os, re, textwrap, subprocess, json, base64, mimetypes, platform, shutil, time
from pathlib import Path
from typing import List, Dict, Optional, TypedDict, Any
from difflib import SequenceMatcher
from urllib.parse import urlparse

# PPT 파싱
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE, PP_PLACEHOLDER

# OpenAI
from openai import OpenAI

# LangChain
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 🌟 v4 최신 고급 기능 Import 🌟
from langchain.agents import create_agent
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# 검색 (RAG)
from tavily import TavilyClient
from langchain_community.tools.tavily_search import TavilySearchResults

# Jupyter 표시용
from IPython.display import Audio, Video, display

# ============================================================
# 🔑 환경변수 로드 (.env 파일에서 자동으로 읽어옵니다)
# ============================================================
from dotenv import load_dotenv
load_dotenv()

LLM_MODEL = os.getenv("LLM_MODEL", "gpt-4o")
TTS_MODEL = os.getenv("TTS_MODEL", "tts-1")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ============================================================
# 🎬 FFmpeg 경로 통일 (v3 개선: 모든 노드에서 동일 변수 사용)
# ============================================================
FFMPEG_CMD = "ffmpeg"
FFPROBE_CMD = "ffprobe"
if platform.system() == "Windows":
    winget_ffmpeg = r"C:\Users\USER\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.1.2-full_build\bin\ffmpeg.exe"
    winget_ffprobe = r"C:\Users\USER\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.1.2-full_build\bin\ffprobe.exe"
    if os.path.exists(winget_ffmpeg):
        FFMPEG_CMD = winget_ffmpeg
    if os.path.exists(winget_ffprobe):
        FFPROBE_CMD = winget_ffprobe

print(f"✅ 환경 설정 완료")
print(f"   LLM: {LLM_MODEL} | TTS: {TTS_MODEL}")
print(f"   FFmpeg: {FFMPEG_CMD}")

C:\Users\USER\AppData\Local\Temp\ipykernel_19200\3497496506.py:25: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.wikipedia.tool import WikipediaQueryRun


✅ 환경 설정 완료
   LLM: gpt-4o | TTS: gpt-4o-mini-tts
   FFmpeg: C:\Users\USER\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.1.2-full_build\bin\ffmpeg.exe


## 🔧 Cell 2. 유틸리티 함수

v2에서 2번 정의되어 있던 `ffprobe_duration` 등을 **1곳에 통합**했습니다.

In [2]:
# ============================================================
# 🔧 유틸리티 함수 (v3 개선: 중복 제거, 통합)
# ============================================================

def clean_text(s):
    """공백/줄바꿈 정리"""
    return re.sub(r"\s+", " ", s).strip()

def split_sents(t: str) -> List[str]:
    """문장 단위 분리"""
    parts = re.split(r'([.?!])', t)
    merged = []
    for i in range(0, len(parts)-1, 2):
        sent = (parts[i] + parts[i+1]).strip()
        if sent:
            merged.append(sent)
    if len(parts) % 2 == 1 and parts[-1].strip():
        merged.append(parts[-1].strip())
    return [s for s in merged if s]

def ffprobe_duration(path: str) -> float:
    """오디오/비디오 파일의 재생 시간(초) 반환"""
    out = subprocess.check_output([
        FFPROBE_CMD, "-v", "error", "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1", path
    ]).decode().strip()
    return float(out)

def img_to_data_url(path: str) -> str:
    """이미지를 base64 data URL로 변환"""
    mime = mimetypes.guess_type(path)[0] or "image/png"
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime};base64,{b64}"

def export_slide_as_png(state: dict, dpi: int = 220) -> dict:
    """PPT 슬라이드 1장을 PNG 이미지로 변환"""
    work_dir = Path(state["work_dir"]).expanduser().resolve()
    work_dir.mkdir(parents=True, exist_ok=True)
    pptx_path = Path(state["pptx_path"]).expanduser().resolve()
    if not pptx_path.exists():
        raise FileNotFoundError(f"PPTX 없음: {pptx_path}")
    idx = int(state.get("slide_index", 0))
    page_no = idx + 1
    out_prefix = work_dir / "slide_img"
    png_path = work_dir / f"{out_prefix.stem}-{page_no}.png"
    env = os.environ.copy()
    env.update({"LANG": "ko_KR.UTF-8", "LC_ALL": "ko_KR.UTF-8"})
    soffice_cmd = "soffice"
    if platform.system() == "Windows":
        if os.path.exists(r"C:\Program Files\LibreOffice\program\soffice.exe"):
            soffice_cmd = r"C:\Program Files\LibreOffice\program\soffice.exe"
    pdf_path = work_dir / f"{pptx_path.stem}.pdf"
    if not pdf_path.exists():
        lo_cmd = [soffice_cmd, "--headless", "--convert-to", "pdf:impress_pdf_Export", "--outdir", str(work_dir), str(pptx_path)]
        if platform.system() != "Windows":
            lo_cmd.insert(2, "-env:UserInstallation=file:///tmp/lo_profile")
        res_pdf = subprocess.run(lo_cmd, capture_output=True, text=True, env=env)
        if res_pdf.returncode != 0:
            raise RuntimeError("PPTX → PDF 변환 실패")
    ppm_cmd = ["pdftoppm", "-f", str(page_no), "-l", str(page_no), "-png", "-r", str(dpi), str(pdf_path), str(out_prefix)]
    subprocess.run(ppm_cmd, capture_output=True, text=True, env=env)
    if png_path.exists():
        state["slide_image"] = str(png_path)
    else:
        print(f"⚠️ 슬라이드 {page_no} PNG 변환 실패")
        state["slide_image"] = ""
    return state

print("✅ 유틸리티 함수 로드 완료")

✅ 유틸리티 함수 로드 완료


## 📋 Cell 3. State 정의

v3에서 추가된 필드: `notes`(발표자 노트), `used_expressions`(반복 방지용 표현 추적)

In [3]:
# ============================================================
# 📋 State 정의 (v3 개선: notes, used_expressions 추가)
# ============================================================

class State(TypedDict, total=False):
    # 입력/기본
    pptx_path: str
    work_dir: str
    prompt: Dict
    slide_index: int
    total_slides: int

    # 추출 산출물
    titles: List[str]
    texts: List[str]
    notes: List[str]              # ⭐ v3 추가: 발표자 노트
    tables: List[List[List[str]]]
    images: List[str]
    slide_image: List[str]
    external_content: Dict[str, List[Dict[str, str]]]
    shape_texts: List[List[str]]
    links: List[List[str]]

    # 생성 산출물
    page_content: str
    script: str
    all_scripts: List[str]
    used_expressions: List[str]   # ⭐ v3 추가: 반복 방지용

    # 미디어 산출물
    audio: str
    video_path: str
    video_paths: List[str]
    final_video: str

print("✅ State 정의 완료")

✅ State 정의 완료


## 🔍 Cell 4. Node 1: PPT 파싱 (`node_parse_all`)

**v3 개선점**: 발표자 노트(Notes) 추출 추가

In [4]:
def node_parse_all(state: State) -> State:
    pptx_path = state["pptx_path"]
    work_dir = state.get("work_dir", "./")

    base_dir = Path(work_dir).expanduser().resolve()
    slides_dir = base_dir / "slides"
    media_dir = base_dir / "media"
    slides_dir.mkdir(parents=True, exist_ok=True)
    media_dir.mkdir(parents=True, exist_ok=True)

    if not os.path.exists(pptx_path):
        print(f"❌ 오류: '{pptx_path}' 파일이 존재하지 않습니다.")
        return state

    prs = Presentation(pptx_path)
    slide_count = len(prs.slides)
    print(f"총 {slide_count}장의 슬라이드를 처리합니다.")

    titles_list, texts_list, notes_list = [], [], []
    tables_list, images_list, snapshots_list = [], [], []
    shape_texts_list, links_list = [], []
    url_pattern = r"https?://[^\s]+"

    for slide_idx, slide in enumerate(prs.slides):
        print(f"\n=== {slide_idx+1}번째 슬라이드 처리 중 ===")

        # 1) PNG 스냅샷 생성
        slide_state = {"pptx_path": pptx_path, "work_dir": str(slides_dir), "slide_index": slide_idx}
        snapshot_path = ""
        try:
            slide_state = export_slide_as_png(slide_state)
            src_png = slide_state.get("slide_image", "")
            dst_png = slides_dir / f"slide_{slide_idx+1}.png"
            if src_png and os.path.exists(src_png):
                os.replace(src_png, dst_png)
                snapshot_path = str(dst_png)
        except Exception as e:
            print(f"  ⚠️ PNG 변환 실패 (텍스트 추출은 계속): {e}")

        # ⭐ v3 추가: 발표자 노트 추출
        slide_note = ""
        try:
            if slide.has_notes_slide:
                notes_tf = slide.notes_slide.notes_text_frame
                if notes_tf and notes_tf.text.strip():
                    slide_note = notes_tf.text.strip()
        except Exception:
            pass

        slide_title, body_texts, shape_texts = "", [], []
        slide_tables, slide_images = [], []
        slide_links = set()

        # 차트 데이터 추출
        for sh in slide.shapes:
            if sh.has_chart:
                try:
                    chart = sh.chart
                    chart_data = ["\n[📊 차트 데이터]"]
                    if chart.has_title and chart.chart_title.has_text_frame:
                        chart_data.append(f"차트 제목: {chart.chart_title.text_frame.text}")
                    series_names = [s.name for s in chart.series]
                    chart_data.append("항목 | " + " | ".join(series_names))
                    if chart.plots:
                        plot = chart.plots[0]
                        categories = [c.label for c in plot.categories]
                        for i, cat in enumerate(categories):
                            row_vals = [str(cat)]
                            for s in chart.series:
                                val = s.values[i] if i < len(s.values) else ""
                                row_vals.append(str(val))
                            chart_data.append(" | ".join(row_vals))
                    body_texts.append("\n".join(chart_data))
                except Exception as e:
                    print(f"  ⚠️ 차트 데이터 추출 실패: {e}")

        # 제목 추출
        for sh in slide.shapes:
            try:
                if sh.is_placeholder and sh.placeholder_format.type == PP_PLACEHOLDER.TITLE:
                    if sh.has_text_frame and sh.text.strip():
                        slide_title = sh.text.strip()
            except:
                pass

        # 도형 텍스트
        def collect_shape_texts(shape):
            if shape.shape_type == MSO_SHAPE_TYPE.GROUP:
                for sub in shape.shapes:
                    collect_shape_texts(sub)
                return
            if shape.shape_type == MSO_SHAPE_TYPE.AUTO_SHAPE and shape.has_text_frame:
                t = shape.text_frame.text.strip()
                if t:
                    shape_texts.append(t)
        for sh in slide.shapes:
            collect_shape_texts(sh)

        # 링크 추출
        for sh in slide.shapes:
            if sh.has_text_frame:
                for p in sh.text_frame.paragraphs:
                    runs_text = "".join(r.text for r in p.runs)
                    found = re.findall(url_pattern, runs_text)
                    if found:
                        slide_links.update(found)
            try:
                if hasattr(sh, "click_action") and sh.click_action.hyperlink.address:
                    slide_links.add(sh.click_action.hyperlink.address)
            except:
                pass

        # 본문 텍스트 (제목/도형/링크 제외)
        for sh in slide.shapes:
            if not sh.has_text_frame:
                continue
            try:
                if sh.is_placeholder and sh.placeholder_format.type == PP_PLACEHOLDER.TITLE:
                    continue
            except:
                pass
            for p in sh.text_frame.paragraphs:
                txt = "".join(r.text for r in p.runs).strip()
                if not txt or txt in shape_texts or re.search(url_pattern, txt):
                    continue
                body_texts.append(txt)

        # 표 추출
        for sh in slide.shapes:
            if sh.shape_type == MSO_SHAPE_TYPE.TABLE:
                tbl = [[cell.text.strip() for cell in row.cells] for row in sh.table.rows]
                slide_tables.append(tbl)

        # 이미지 추출 (그룹/숨겨진 이미지 포함)
        def extract_images_from_shape(shape, img_list):
            if shape.shape_type == MSO_SHAPE_TYPE.GROUP:
                for sub_shape in shape.shapes:
                    extract_images_from_shape(sub_shape, img_list)
                return
            if hasattr(shape, "image"):
                try:
                    ext = shape.image.ext
                except:
                    ext = "png"
                img_name = f"slide{slide_idx+1}_img_{len(img_list)+1}.{ext}"
                img_path = media_dir / img_name
                with open(img_path, "wb") as f:
                    f.write(shape.image.blob)
                img_list.append(str(img_path))
        for sh in slide.shapes:
            extract_images_from_shape(sh, slide_images)

        # 리스트에 저장
        titles_list.append(slide_title)
        texts_list.append("\n".join(body_texts).strip())
        notes_list.append(slide_note)
        tables_list.append(slide_tables)
        images_list.append(slide_images)
        snapshots_list.append(snapshot_path)
        shape_texts_list.append(list(dict.fromkeys(shape_texts)))
        links_list.append(list(slide_links))

        print(f"  → 제목:{slide_title}, 본문:{len(body_texts)}, 노트:{'있음' if slide_note else '없음'}, 표:{len(slide_tables)}, 이미지:{len(slide_images)}")

    state["total_slides"] = slide_count
    state["titles"] = titles_list
    state["texts"] = texts_list
    state["notes"] = notes_list
    state["tables"] = tables_list
    state["images"] = images_list
    state["slide_image"] = snapshots_list
    state["shape_texts"] = shape_texts_list
    state["links"] = links_list
    return state

print("✅ node_parse_all 정의 완료")

✅ node_parse_all 정의 완료


## 🧪 Cell 5. [테스트] node_parse_all

In [5]:
# test_state = {
#     "pptx_path": "./data/sample.pptx",
#     "work_dir": "./output_v4_test"
# }
# result_state = node_parse_all(test_state)

# print("\n" + "="*50)
# print("🎯 [첫 번째 슬라이드 추출 결과]")
# print("="*50)
# idx = 0
# print(f"▶ 제목: {result_state['titles'][idx]}")
# print(f"▶ 본문: {result_state['texts'][idx][:200]}...")
# print(f"▶ 노트: {result_state['notes'][idx][:200] if result_state['notes'][idx] else '(없음)'}")
# print(f"▶ 표: {len(result_state['tables'][idx])}개")
# print(f"▶ 이미지: {len(result_state['images'][idx])}개")

## 🔍 Cell 6. 검색 점수 함수 + Tavily 검색

v3 개선: 불필요한 `print(results_sorted)` 디버그 로그 제거

In [6]:
# ============================================================
# 🔍 검색 점수 계산 함수 + Tavily 검색
# ============================================================

DOMAIN_TRUST = {
    "microsoft.com": 0.95, "learn.microsoft.com": 0.98,
    "docs.python.org": 0.98, "wikipedia.org": 0.9,
    "mozilla.org": 0.9, "naver.com": 0.8,
    "cloud.naver.com": 0.9, "kakao.com": 0.8,
}

EXCLUDE_DOMAINS = [
    "blog.naver.com", "m.blog.naver.com", "tistory.com",
    "brunch.co.kr", "medium.com", "velog.io",
    "kin.naver.com", "reddit.com", "youtube.com"
]

def _norm_text(s: str) -> str:
    s = (s or "").lower()
    s = re.sub(r"[^가-힣a-z0-9\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def similarity_score(query: str, snippet: str) -> float:
    q, t = _norm_text(query), _norm_text(snippet)
    if not q or not t:
        return 0.0
    q_tokens, t_tokens = set(q.split()), set(t.split())
    overlap = len(q_tokens & t_tokens) / len(q_tokens) if q_tokens and t_tokens else 0.0
    seq_sim = SequenceMatcher(None, q, t).ratio()
    return (overlap + seq_sim) / 2.0

def domain_score(domain: str) -> float:
    domain = (domain or "").lower()
    for key, score in DOMAIN_TRUST.items():
        if key in domain:
            return score
    if not domain:
        return 0.4
    return max(0.45, min(0.5 + min(len(domain) / 50.0, 0.15), 0.65))

def content_score(snippet: str, max_len: int = 400) -> float:
    if not snippet:
        return 0.0
    return max(0.1, min(len(snippet) / max_len, 1.0))

def tavily_search(title: str, num: int = 4) -> list[dict]:
    query = f"{title} " + " ".join([f"-site:{d}" for d in EXCLUDE_DOMAINS])
    candidate_k = max(num * 3, num + 2)
    res_tool = TavilySearchResults(
        max_results=candidate_k, search_depth="basic", topic="general",
        exclude_domains=EXCLUDE_DOMAINS, include_answer=False, include_raw_content=False,
    )
    data = res_tool.invoke(query) or []
    results, seen_urls = [], set()
    for item in data:
        url = item.get("url", "")
        if not url or url in seen_urls:
            continue
        domain = urlparse(url).netloc
        if any(ex in domain for ex in EXCLUDE_DOMAINS):
            continue
        seen_urls.add(url)
        snippet_res = item.get("content", "") or ""
        s_sim = similarity_score(title, snippet_res)
        s_dom = domain_score(domain)
        s_cont = content_score(snippet_res)
        score = 0.5 * s_sim + 0.3 * s_dom + 0.2 * s_cont
        results.append({
            "title": item.get("title", ""), "url": url,
            "snippet": snippet_res, "domain": domain,
            "score": round(score, 4),
        })
    results_sorted = sorted(results, key=lambda x: x["score"], reverse=True)
    return results_sorted[:num]

print("✅ 검색 함수 로드 완료")

✅ 검색 함수 로드 완료


## 🌐 Cell 7. Node 2: 외부 검색 (`node_tool_search`)

v3 개선: 불필요한 state 전체 출력 제거, 로그 간소화

In [7]:
def node_tool_search(state: dict) -> dict:
    """외부 검색 노드 (v4: 최신 LangChain create_agent)"""
    print("Node 2: 외부 검색(tool_search) v4 Agent 실행 ---")
    state["external_content"] = {"queries": [], "summaries": [], "references": []}

    idx = state.get("slide_index", 0)
    title = state.get("titles", [])[idx] if idx < len(state.get("titles", [])) else ""
    texts = state.get("texts", [])[idx] if idx < len(state.get("texts", [])) else ""

    if not title and not texts:
        print("  검색할 내용이 없습니다.")
        return state

    print(f"  슬라이드 {idx+1} | 제목: {title}")

    # 1. 툴 준비
    wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(lang="ko"))
    
    from langchain_core.tools import StructuredTool
    def tavily_search_tool(query: str) -> str:
        res = tavily_search(query, num=3)
        return str(res)
        
    tavily_tool = StructuredTool.from_function(
        func=tavily_search_tool,
        name="tavily_search",
        description="Search the web for current information, statistics, or general knowledge. Input should be a search query."
    )
    tools = [tavily_tool, wiki]

    # 2. 최신 create_agent 구성 (AgentExecutor 대체)
    llm = ChatOpenAI(model=LLM_MODEL, temperature=0)
    system_prompt = (
        "당신은 강의 자료를 만들기 위해 전문 자료를 검색하는 AI 연구원입니다."
        "주어진 슬라이드 내용을 보고 필요한 배경지식, 최신 동향, 전문 용어의 뜻을 알아내기 위해 검색 도구를 사용하세요."
        "검색 후, 슬라이드를 더 풍부하게 설명할 수 있는 요약 정보를 3~4문장으로 작성해주세요."
    )

    agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt=system_prompt
    )

    print("  [Agent] 최신 방식(create_agent)으로 스스로 검색 판단 및 수행 중...")
    try:
        inputs = {"messages": [{"role": "user", "content": f"""제목: {title}
내용: {texts[:200]}
이 슬라이드 내용을 강의로 설명하기 위해 필요한 추가 정보를 검색하고 요약해 주세요."""}]}
        result = agent.invoke(inputs)
        
        # 최신 Agent는 마지막 AIMessage에 최종 응답을 담습니다.
        summary_text = result["messages"][-1].content
        print("  ✅ Agent 검색/요약 완료")
        
        state["external_content"]["summaries"].append({
            "text": summary_text,
            "source": "AI Agent Research",
            "score": 1.0
        })
    except Exception as e:
        print(f"  ⚠️ Agent 검색 실패: {e}")

    return state

print("✅ node_tool_search v4 정의 완료")

✅ node_tool_search v4 정의 완료


## 📝 Cell 8. Node 3: 요약 노트 생성 (`node_generate_page_content`)

v3 개선: 발표자 노트(notes)를 프롬프트에 포함

In [8]:
class SlideNote(BaseModel):
    summary: str = Field(description="슬라이드의 내용을 바탕으로 한 4~6문장의 줄글 요약 (번호 매기기 절대 금지)")
    key_terms: list[str] = Field(description="이 슬라이드의 핵심 키워드 딱 3개")
    difficulty: str = Field(description="초급, 중급, 고급 중 택 1")

def node_generate_page_content(state: State) -> State:
    """슬라이드 데이터 + 외부 자료 + 이미지 → LLM 요약 노트 (v4: Structured Output)"""
    print("Node 3: 요약 노트 생성(gen_page_content) v4 Pydantic 실행 ---")

    idx = int(state.get("slide_index", 0))
    title = clean_text(str(state.get("titles", [])[idx])) if idx < len(state.get("titles", [])) else ""
    texts = clean_text(str(state.get("texts", [])[idx])) if idx < len(state.get("texts", [])) else ""
    notes_all = state.get("notes", [])
    slide_note = notes_all[idx] if idx < len(notes_all) else ""

    ext = state.get("external_content", {}) or {}
    ext_summary = ""
    if ext.get("summaries"):
        ext_summary = ext["summaries"][0]["text"]

    parser = PydanticOutputParser(pydantic_object=SlideNote)
    format_instructions = parser.get_format_instructions()

    system_instruction = f"""당신은 PPT 내용과 외부 자료를 바탕으로 강연 대본용 핵심 노트를 정리하는 전문 강사입니다.
[지시사항]
1. 과장 금지, 객관적으로 요약할 것.
2. 번호 매기기나 불릿 금지. 줄글 형태로 자연스럽게 작성.

[출력 형식]
{format_instructions}"""

    user_data = f"""제목: {title}
[텍스트]
{texts}
[발표자 노트]
{slide_note}
[외부 조사 내용]
{ext_summary}"""

    messages = [SystemMessage(content=system_instruction), HumanMessage(content=user_data)]
    print(f"  [LLM 요청] Pydantic 구조화 파싱 진행 중...")

    llm = ChatOpenAI(model=LLM_MODEL, temperature=0.1)
    
    try:
        response = llm.invoke(messages)
        parsed_note = parser.invoke(response)
        state["page_content"] = parsed_note.summary
        print(f"  ✅ 요약 노트 생성 완료: 난이도({parsed_note.difficulty}), 키워드({', '.join(parsed_note.key_terms)})")
    except Exception as e:
        print(f"  ⚠️ 파싱 실패: {e}")
        state["page_content"] = response.content if hasattr(response, 'content') else str(response)

    return state

print("✅ node_generate_page_content v4 정의 완료")

✅ node_generate_page_content v4 정의 완료


## 🎤 Cell 9. Node 4: 강의 스크립트 생성 (`node_generate_script`)

### ⚠️ v3 핵심 개선 사항 (v2 대비)
1. **금지 표현 4개 → 15개로 대폭 확대** ("여러분", "이게 무슨 뜻일까요" 등)
2. **프롬프트에 반복 방지 지시 명시적 추가**: "이전에 사용한 표현을 절대 반복하지 마세요"
3. **`used_expressions` 추적**: 이전 스크립트에서 쓴 시작 문장을 추출하여 금지 목록에 추가
4. **글자 수 검증**: 목표 범위 벗어나면 자동 재생성 (1회)

In [9]:
# ============================================================
# 🎤 v3 개선: 금지 표현 확대 + 반복 방지 + 글자 수 검증
# ============================================================

# ⭐ v3: 금지 표현 리스트 대폭 확대 (v2는 4개 → v3는 15개)
def node_generate_script(state: dict) -> dict:
    """v4 개선: 최신 State 기반 Memory + Self-Reflection (이중 검열)"""
    print("Node 4: 강의 스크립트 생성(gen_script) v4 Memory+Reflection 실행 ---")

    all_titles = state.get("titles", [])
    idx = int(state.get("slide_index", 0))
    current_title = all_titles[idx] if idx < len(all_titles) else f"슬라이드 {idx+1}"
    page_content = clean_text(state.get("page_content", ""))
    work_dir = state.get("work_dir", "./")

    if not page_content.strip():
        page_content = f"{current_title}의 핵심 개념을 설명해 주세요."

    prompt_data = state.get("prompt", {})
    final_tone = prompt_data.get("tone", "친절하고 명료한 톤") if isinstance(prompt_data, dict) else prompt_data
    final_style = prompt_data.get("style", "자연스럽고 설명적인 말투") if isinstance(prompt_data, dict) else "자연스럽고 설명적인 말투"
    final_sec = prompt_data.get("target_duration_sec", 50) if isinstance(prompt_data, dict) else 50

    min_chars = int(final_sec * 9)
    max_chars = int(final_sec * 13)

    llm = ChatOpenAI(model=LLM_MODEL, temperature=0.3)
    
    # 🌟 최신 LangGraph 방식 메모리 관리: 외부 모듈 없이 State의 all_scripts를 직접 활용
    # 이전 3개의 스크립트를 가져와서 히스토리로 구성합니다. (ConversationBufferWindowMemory 대체)
    all_scripts = state.get("all_scripts", [])
    recent_scripts = all_scripts[-3:] if all_scripts else []
    history_str = "".join([f"이전 스크립트 {i+1}: {s}" for i, s in enumerate(recent_scripts)])
    
    draft_prompt = ChatPromptTemplate.from_messages([
        ("system", """당신은 전문 강사입니다. 요약 텍스트를 자연스러운 '말하는 글'로 변환하세요.
[규칙] '여러분'은 절대 쓰지 마세요. 이번 슬라이드에서는, 다음으로 넘어가 등의 뻔한 말도 금지합니다."""),
        ("human", """이전 대화 내역 (참고용 흐름): {history}

# 현재 요약:
{page_content}

# 조건:
톤: {tone}
스타일: {style}
위 요약을 {min_chars}~{max_chars}자 길이의 말하는 스크립트 초안으로 작성하세요.""")
    ])
    
    draft_chain = draft_prompt | llm | StrOutputParser()
    
    print("  [LLM] 1차 스크립트(Draft) 생성 중...")
    draft_script = draft_chain.invoke({
        "history": history_str if history_str else "(첫 슬라이드입니다)",
        "page_content": page_content,
        "tone": final_tone, "style": final_style,
        "min_chars": min_chars, "max_chars": max_chars
    })
    
    review_prompt = ChatPromptTemplate.from_messages([
        ("system", """당신은 아주 깐깐한 스크립트 검수자입니다.
아래 초안(Draft)을 읽고, 다음 [절대 금지 표현]이 포함되어 있다면 그 문장을 수정하거나 삭제해서 다시 작성하세요.

[절대 금지 표현]
- "여러분"
- "이번 슬라이드에서는", "지금 보시는 슬라이드는"
- "이게 무슨 뜻일까요?", "여기 집중해 보세요!"
- "~아시겠죠?", "자, 이제"

반드시 수정된 최종 스크립트만 출력하세요 (부연 설명 금지)."),
        ("human", "초안(Draft):
{draft}

위 초안을 검수하고 금지된 표현을 완벽히 제거한 최종본을 출력하세요.""")
    ])
    
    review_chain = review_prompt | llm | StrOutputParser()
    print("  [LLM] 2차 검열(Self-Reflection) 진행 중...")
    final_script = review_chain.invoke({"draft": draft_script}).strip()

    script_len = len(final_script)
    if script_len < min_chars:
        print(f"  ⚠️ 스크립트가 너무 짧습니다 ({script_len}자 < {min_chars}자)")
    elif script_len > max_chars:
        print(f"  ⚠️ 스크립트가 너무 깁니다 ({script_len}자 > {max_chars}자)")

    state["script"] = final_script
    if "all_scripts" not in state:
        state["all_scripts"] = []
    state["all_scripts"].append(final_script)

    os.makedirs(work_dir, exist_ok=True)
    out_path = os.path.join(work_dir, f"script_{idx}.txt")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(final_script)

    print(f"  ✅ 검열 완료 및 스크립트 저장 ({script_len}자)")
    return state

print("✅ node_generate_script v4 정의 완료")

✅ node_generate_script v4 정의 완료


## 🧪 Cell 10. [테스트] 요약 + 스크립트 (Node 2~4 연속 실행)

In [10]:
# print("🔍 [테스트] Node 2 → 3 → 4 연속 실행")
# print("-" * 50)

# # Node 2: 외부 검색
# search_state = node_tool_search(result_state)

# # Node 3: 요약 노트
# content_state = node_generate_page_content(search_state)
# print(f"\n📝 요약 노트:\n{content_state['page_content'][:300]}...")

# # Node 4: 스크립트
# script_state = node_generate_script(content_state)

# print("\n" + "=" * 50)
# print("🎯 [생성된 스크립트]")
# print("=" * 50)
# print(script_state["script"])

## 🎧 Cell 11. Node 5: TTS 음성 변환 (`node_tts`)

v3 개선: `FFMPEG_CMD`/`FFPROBE_CMD` 통일 변수 사용

In [11]:
def node_tts(state: dict) -> dict:
    """교육용 강의 스크립트를 TTS로 변환"""
    print("\n--- Node 5: TTS 변환 실행 ---")

    script = state.get("script", "").strip()
    prompt = state.get("prompt", {}) or {}
    work_dir = state.get("work_dir", "./")
    slide_idx = int(state.get("slide_index", 0))

    EDUCATION_VOICES = {
        "부드러운 설명형": "nova", "교수님 톤": "alloy",
        "친절한 튜토리얼": "shimmer", "명확한 설명형": "onyx",
    }
    raw_voice = prompt.get("voice", "부드러운 설명형")
    voice = EDUCATION_VOICES.get(raw_voice, "nova")
    speed = float(prompt.get("speed", 1.0))

    if not script:
        script = "현재 슬라이드에는 설명할 내용이 준비되어 있지 않습니다."

    os.makedirs(work_dir, exist_ok=True)
    raw_path = os.path.join(work_dir, f"tts_raw_slide{slide_idx}.mp3")
    final_path = os.path.join(work_dir, f"tts_slide{slide_idx}_{speed}x.mp3")

    try:
        print(f"  [TTS] 보이스('{voice}')로 음성 생성 중...")
        response = client.audio.speech.create(model=TTS_MODEL, voice=voice, input=script, response_format="mp3")
    except Exception as e:
        print(f"  [오류] TTS 생성 실패: {e}")
        print("  [재시도] 기본 보이스 'nova'로 재생성")
        response = client.audio.speech.create(model=TTS_MODEL, voice="nova", input=script, response_format="mp3")

    with open(raw_path, "wb") as f:
        f.write(response.read())

    # ⭐ v3 개선: FFMPEG_CMD 통일 변수 사용
    if speed != 1.0 and shutil.which(FFMPEG_CMD):
        print(f"  [FFmpeg] {speed}배속 변환 중...")
        cmd = [FFMPEG_CMD, "-y", "-i", raw_path, "-filter:a", f"atempo={speed}", final_path]
        subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        if os.path.exists(raw_path):
            os.remove(raw_path)
    else:
        final_path = raw_path

    try:
        duration = ffprobe_duration(final_path)
        print(f"  [TTS] 최종 오디오 길이: {round(duration, 2)}초")
    except:
        print("  ⚠️ 오디오 길이 계산 불가")

    state["audio"] = final_path
    print(f"  ✅ 음성 파일 저장 → {final_path}")
    return state

print("✅ node_tts 정의 완료")

✅ node_tts 정의 완료


## 🎬 Cell 12. Node 6: 영상 제작 (`node_make_video`)

In [12]:
def node_make_video(state: dict) -> dict:
    """슬라이드 PNG + TTS 음성 → MP4 클립"""
    print("\n--- Node 6: 영상 생성(make_video) 실행 ---")

    slide_imgs = state.get("slide_image", [])
    audio_path = state.get("audio", "")
    work_dir = state.get("work_dir", "./")
    slide_index = int(state.get("slide_index", 0))

    if not slide_imgs or slide_index >= len(slide_imgs):
        print(f"  [경고] 슬라이드 이미지 없음 → index={slide_index}")
        return state
    if not os.path.exists(audio_path):
        print(f"  [경고] 오디오 파일 없음 → {audio_path}")
        return state

    image_path = slide_imgs[slide_index]
    if not os.path.exists(image_path):
        print(f"  [경고] 이미지 파일 없음 → {image_path}")
        return state

    os.makedirs(work_dir, exist_ok=True)
    out_mp4 = os.path.join(work_dir, f"slide{slide_index+1}_lecture.mp4")

    try:
        duration = ffprobe_duration(audio_path)
    except Exception as e:
        print(f"  [경고] ffprobe 실패 → 기본 5초 (오류: {e})")
        duration = 5

    ffmpeg_cmd = [
        FFMPEG_CMD, "-y", "-loop", "1", "-i", image_path, "-i", audio_path,
        "-t", str(duration),
        "-vf", "scale=1920:1080:force_original_aspect_ratio=decrease,pad=1920:1080:(ow-iw)/2:(oh-ih)/2:color=black",
        "-c:v", "libx264", "-c:a", "aac", "-b:a", "192k", "-pix_fmt", "yuv420p",
        out_mp4
    ]

    print(f"  [FFmpeg] 슬라이드 {slide_index+1} 렌더링 중...")
    result = subprocess.run(ffmpeg_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    if result.returncode != 0:
        print("  [오류] 렌더링 실패 → 1회 재시도")
        retry = subprocess.run(ffmpeg_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        if retry.returncode != 0:
            print("  [치명] 영상 생성 실패. 건너뜁니다.")
            return state

    state["video_path"] = out_mp4
    print(f"  ✅ 영상 생성 → {out_mp4}")
    return state

print("✅ node_make_video 정의 완료")

✅ node_make_video 정의 완료


## 🔄 Cell 13. Node 7~9: 반복/종료/합치기

In [13]:
def node_accumulate_and_step(state: dict) -> dict:
    """영상 누적 + 다음 슬라이드 이동"""
    current_idx = state.get("slide_index", 0)
    total = state.get("total_slides", 1)
    current_video = state.get("video_path", None)

    if "video_paths" not in state or not isinstance(state["video_paths"], list):
        state["video_paths"] = []

    if current_video and os.path.exists(current_video):
        if current_video not in state["video_paths"]:
            state["video_paths"].append(current_video)
        print(f"  슬라이드 {current_idx+1} 완료 → {current_video}")
    else:
        print(f"  슬라이드 {current_idx+1} 영상 생성 실패")
        state.setdefault("failed_slides", []).append(current_idx)

    state["slide_index"] = current_idx + 1
    progress = (state["slide_index"] / total) * 100
    print(f"  진행률: {state['slide_index']}/{total} ({progress:.1f}%)")
    return state

def router_continue_or_done(state: dict) -> str:
    current = state.get("slide_index", 0)
    total = state.get("total_slides", 1)
    if current >= total:
        print("\n🎉 모든 슬라이드 처리 완료!")
        return "done"
    print(f"\n➡️ 다음 슬라이드: {current+1}/{total}")
    return "continue"

def node_concat(state: dict) -> dict:
    """모든 영상을 하나로 합침"""
    video_paths = state.get("video_paths", [])
    work_dir = state.get("work_dir", "./")

    if not video_paths:
        print("❗ 합칠 영상이 없습니다.")
        return state

    print(f"총 {len(video_paths)}개 영상 병합 시작")
    video_paths = sorted(video_paths, key=lambda x: int(re.findall(r"slide(\d+)", x)[0]))
    final_video = os.path.join(work_dir, "final_lecture.mp4")

    input_cmd, filter_inputs = [], ""
    for i, path in enumerate(video_paths):
        input_cmd += ["-i", path]
        filter_inputs += f"[{i}:v][{i}:a]"
    filter_cmd = f"{filter_inputs}concat=n={len(video_paths)}:v=1:a=1[outv][outa]"

    cmd = [
        FFMPEG_CMD, "-y", *input_cmd,
        "-filter_complex", filter_cmd,
        "-map", "[outv]", "-map", "[outa]",
        "-c:v", "libx264", "-c:a", "aac",
        "-pix_fmt", "yuv420p", "-movflags", "+faststart",
        final_video
    ]

    print("  FFmpeg 병합 중...")
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if result.returncode != 0:
        print("[❌] 영상 병합 실패")
        print(result.stderr.decode())
        return state

    try:
        duration = ffprobe_duration(final_video)
    except:
        duration = 0.0
    size_mb = os.path.getsize(final_video) / (1024 * 1024)

    print("🎉 최종 강의 영상 생성 완료!")
    print(f"  경로: {final_video}")
    if duration > 0:
        print(f"  재생 시간: {duration:.1f}초")
    print(f"  파일 크기: {size_mb:.2f} MB")

    state["final_video"] = final_video
    return state

print("✅ accumulate + router + concat 정의 완료")

✅ accumulate + router + concat 정의 완료


## 🏗️ Cell 14. LangGraph 빌드

In [14]:
from langgraph.graph import StateGraph, END

builder = StateGraph(State)

# 노드 등록
builder.add_node("parse_ppt", node_parse_all)
builder.add_node("tool_search", node_tool_search)
builder.add_node("gen_page_content", node_generate_page_content)
builder.add_node("gen_script", node_generate_script)
builder.add_node("tts", node_tts)
builder.add_node("make_video", node_make_video)
builder.add_node("accumulate", node_accumulate_and_step)
builder.add_node("concat", node_concat)

# 엣지 연결
builder.set_entry_point("parse_ppt")
builder.add_edge("parse_ppt", "tool_search")
builder.add_edge("tool_search", "gen_page_content")
builder.add_edge("gen_page_content", "gen_script")
builder.add_edge("gen_script", "tts")
builder.add_edge("tts", "make_video")
builder.add_edge("make_video", "accumulate")
builder.add_conditional_edges("accumulate", router_continue_or_done, {"continue": "tool_search", "done": "concat"})
builder.add_edge("concat", END)

# 컴파일
graph_app = builder.compile()
print("✅ LangGraph 빌드 완료!")

✅ LangGraph 빌드 완료!


## 🚀 Cell 15. 전체 파이프라인 실행 테스트

In [16]:
from pathlib import Path

TEST_PPTX_PATH = "./data/sample.pptx"
base_name = Path(TEST_PPTX_PATH).stem
FINAL_WORK_DIR = f"./output_v4_{base_name}"

initial_state = {
    "pptx_path": TEST_PPTX_PATH,
    "work_dir": FINAL_WORK_DIR,
    "prompt": {
        "tone": "에너지가 넘치고 자신감 있는 1타 강사 톤",
        "style": "딱딱한 설명 대신 쉬운 비유와 실생활 예시를 적극 활용하세요. 매 슬라이드마다 다른 방식으로 시작하고 마무리하세요.",
        "target_duration_sec": 70,
        "voice": "친절한 튜토리얼",
        "speed": 1.15
    },
    "slide_index": 0,
}

print("🚀 전체 파이프라인 실행 시작!")
print("=" * 50)

final_state = None
for output in graph_app.stream(initial_state, {"recursion_limit": 100}):
    for key, value in output.items():
        final_state = value

print("\n" + "=" * 50)
print("🎉 파이프라인 실행 완료!")

if final_state and final_state.get("final_video"):
    print(f"📁 최종 영상: {final_state['final_video']}")
    display(Video(final_state["final_video"], embed=True, width=800))
else:
    print("⚠️ 최종 영상이 생성되지 않았습니다.")

🚀 전체 파이프라인 실행 시작!
❌ 오류: './data/sample.pptx' 파일이 존재하지 않습니다.
Node 2: 외부 검색(tool_search) v4 Agent 실행 ---
  검색할 내용이 없습니다.
Node 3: 요약 노트 생성(gen_page_content) v4 Pydantic 실행 ---
  [LLM 요청] Pydantic 구조화 파싱 진행 중...
  ✅ 요약 노트 생성 완료: 난이도(초급), 키워드(발표 주제, 핵심 메시지, 외부 조사)
Node 4: 강의 스크립트 생성(gen_script) v4 Memory+Reflection 실행 ---
  [LLM] 1차 스크립트(Draft) 생성 중...


KeyboardInterrupt: 